# 00 - Raw Dataset Exploration

**Goal**: Validate the raw Kaggle dataset before running the ETL pipeline.

**Source**: [LinkedIn Job Postings Dataset](https://www.kaggle.com/datasets/arshkon/linkedin-job-postings) (Kaggle)

**License**: CC BY-SA 4.0

**Data Ingestion Note (Files Kept vs. Discarded)**:

The original Kaggle `archive.zip` contains additional tables that were intentionally excluded to align with our MVP scope. 
- ✅ **Kept (Core)**: `postings.csv` (core texts), `job_skills.csv` & `skills.csv` (job schemas), `salaries.csv` (dashboard metrics), and `companies.csv` (profile evaluator context).
- ❌ **Discarded (Out of Scope)**: `benefits.csv` (irrelevant for tech stack intelligence), `industries.csv` / `job_industries.csv` / `company_industries.csv` (we rely on title heuristics to filter the Tech sector, making these broad labels redundant), `company_specialities.csv`, and `employee_counts.csv` (excessive granularity).

---

**Questions to answer**:
1. How many job postings are there?
2. Which files does the dataset contain and what columns?
3. Are the key columns (title, description, skills) populated enough to be useful?
4. How many postings match tech-sector keywords? Is the sample large enough?


In [ ]:
from pathlib import Path
import pandas as pd

RAW_DIR = Path("../data/raw")

# List available raw files
print("Raw files:")
for f in sorted(RAW_DIR.iterdir()):
    size_mb = f.stat().st_size / 1_048_576
    print(f"  {f.name:<20} {size_mb:>5.1f} MB")

## 1. Postings — main table

In [ ]:
df = pd.read_csv(RAW_DIR / "postings.csv", usecols=["job_id", "title", "description"])
print(f"Total postings: {len(df):,}")
df.head(4)

In [ ]:
# Null check on critical columns
print("Null values:")
print(df.isnull().sum())
print("\nPostings with both title AND description:", df.dropna(subset=["title", "description"]).shape[0])

## 2. Tech sector coverage

Estimate how many postings would pass the ETL tech filter.

In [ ]:
# Broad tech keywords for title
TECH_TITLE_SAMPLE = [
    r"\bSoftware\b", r"\bData\b", r"\bML\b", r"\bAI\b", r"\bDevOps\b",
    r"\bBackend\b", r"\bFrontend\b", r"\bCloud\b", r"\bCybersecurity\b",
    r"\bMachine Learning\b", r"\bData Scientist\b", r"\bData Engineer\b",
    r"\bSoftware Engineer\b", r"\bPlatform Engineer\b", r"\bQA\b",
]
pattern = "|".join(TECH_TITLE_SAMPLE)

tech_mask = df["title"].str.contains(pattern, case=False, na=False, regex=True)
print(f"Postings with tech title keywords: {tech_mask.sum():,} ({tech_mask.mean()*100:.1f}%)")
print("\nSample tech job titles:")
df[tech_mask]["title"].value_counts().head(20)

## 3. Skills table

In [ ]:
df_skills = pd.read_csv(RAW_DIR / "job_skills.csv")
df_skill_names = pd.read_csv(RAW_DIR / "skills.csv")

print(f"job_skills rows: {len(df_skills):,}")
print(f"skills (lookup) rows: {len(df_skill_names):,}")

# How many postings have at least one skill
postings_with_skills = df_skills["job_id"].nunique()
print(f"\nPostings with at least one skill: {postings_with_skills:,} ({postings_with_skills/len(df)*100:.1f}%)")

## 4. Salaries table

In [ ]:
df_salaries = pd.read_csv(RAW_DIR / "salaries.csv", usecols=["job_id", "max_salary", "pay_period"])
print(f"salaries rows: {len(df_salaries):,}")
print(f"Postings with salary info: {df_salaries['max_salary'].notna().sum():,}")
print("\nPay period distribution:")
print(df_salaries["pay_period"].value_counts())

## 5. Go / No-Go decision

| Check | Minimum | Result |
|---|---|---|
| Total postings | > 50,000 | ✅ |
| Postings with title + description | > 40,000 | ✅ |
| Tech-sector postings (title filter) | > 5,000 | ✅ |
| Postings with skills | > 10,000 | ✅ |
